# Feature Engineering

### NFL Big Data Bowl 2027

by Chelle Davies

In [1]:
%pip install pandas numpy matplotlib seaborn plotly scipy


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display
import scipy
import scipy.stats as stats

In [ ]:
# Load all CSV files in the "nfl-big-data-bowl-2027-data" directory into a dictionary of DataFrames

file_path = "nfl-big-data-bowl-2027-data"

dataframes = {
    os.path.splitext(filename)[0]: pd.read_csv(os.path.join(file_path, filename))
    for filename in os.listdir(file_path)
    if filename.lower().endswith(".csv")
}

list(dataframes)

In [ ]:
for name, df in dataframes.items():
    print(f"DataFrame: {name}, Shape: {df.shape}")
    print(df.head())

---

#### Hypothesis 1: Higher mean combine acceleration is associated with more sacks per game.

#### Hypothesis 2: Greater combine acceleration variability is associated with more sacks per game.

Pass rush success relies on two distinct traits: explosive burst (mean capacity) and deceptive pacing/sudden redirection (variability).

- `top_decile_mean_accel`: The mean of the top 10% highest acceleration frames. Isolates the player's peak capacity rather than diluting it with baseline tracking frames.
- `accel_standard_deviation`: The standard deviation of acceleration across a complete drill. Directly tests the variability hypothesis, measuring a player's ability to violently modulate speed.
- `accel_coefficient_of_variation`: Calculated as `accel_standard_deviation` / `overall_mean_accel`. This normalizes variability against absolute speed, preventing elite athletes from skewing the variance metric.
- `jerk_variance`: The variance of the first derivative of acceleration. Captures how erratic or sudden the player's force adjustments are, representing a non-linear, unpredictable pass rush.


#### Hypothesis 3: Higher mean combine acceleration is associated with more receiving yards per game.

For receivers and tight ends, mean acceleration translates to vertical separation and explosive speed out of breaks.

- `break_window_mean_accel`: The mean acceleration in the 1.0-second window immediately following a major drop in velocity (the top of a route break).
- `linear_strides_mean_accel`: The mean acceleration during pure linear phases (e.g., frames 10 to 30 of the 40-yard dash). Measures sustained acceleration downfield.
- `accel_efficiency_ratio`: The ratio of forward-vector acceleration (`a_{x}`) to lateral acceleration (`a_{y}`). Higher ratios mean less wasted energy during downfield acceleration.


#### Hypothesis 4: Combine movement metrics improve held-out-year predictions beyond position and traditional combine results.

The aim here is to prevent leakage and isolate incremental feature importance.

- The Baseline Feature Group: position_encoded (one-hot), official_40_time, official_3_cone_time, official_shuttle_time, height, weight.
- The Tracking Feature Group: The engineered sensor metrics detailed above (top_decile_mean_accel, accel_coefficient_of_variation, etc.).

______

#### Framework for Model Evaluation

| Hypothesis | Methodology | Strategy |
| --- | --- | --- |
| Hypothesis 1 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 2 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 3 | Feature-Level Significance | Use a tree-based ensemble (e.g., LightGBM or XGBoost) to extract SHAP (SHapley Additive exPlanations) values. Check if accel_coefficient_of_variation and top_decile_mean_accel show significant, positive SHAP main effects against their respective targets. |
| Hypothesis 4 | Model-Level Incremental Lift | Root Mean Squared Error (RMSE) and R² (Delta R² [the full model's R² minus the baseline model's R²] must be greater than 0.) |

##### Steps:
1. Train Baseline Model (Position + Traditional Metrics)
2. Train Full Model (Baseline + 10 Hz Engineered Tracking Features)
3. Evaluate Incremental Lift on a Held-Out Year